# Diagnostic 3: partial freeze, shared head (CLIP BASED)

In [1]:
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/dissertation')
else:
    PROJECT_ROOT = Path('../../..').resolve()

sys.path.insert(0, str(PROJECT_ROOT))

print(f"IN_COLAB={IN_COLAB}  REPO_ROOT={PROJECT_ROOT}")

Mounted at /content/drive
IN_COLAB=True  REPO_ROOT=/content/drive/MyDrive/dissertation


In [2]:
import json
import pandas as pd
import numpy as np

from src import dataloader as data
from src import metrics
from src import train
from src.models.classical_baseline import HORIZONS
from src.models.multimodal_residual import load_frozen_baseline
from src.models.diagnostic_partial_freeze import PartiallyFrozenSharedHeadPredictor

SEEDS = [7, 14, 21, 28, 35, 42, 49, 56, 63, 70]
ENCODER_NAME = 'clip'
IMG_FEAT_DIM = 768
IMG_PROJ_DIM = 4

EXPERIMENT_DIR = PROJECT_ROOT / 'experiments' / '4_diagnostic' / '3_diagnostic'
EXP2_CHECKPOINT_DIR = PROJECT_ROOT / 'experiments' / '3_initial_multimodal' / 'multimodal' / 'checkpoints' / 'gated_cgm_time'
RESULTS_DIR = EXPERIMENT_DIR / 'results' / 'diagnostic_3_clip'
SUMMARY_PATH = RESULTS_DIR / 'diagnostic3_partial_freeze_shared_head_clip_summary.json'

In [3]:
stats = data.load_cgm_stats()
train_loader = data.load_multimodal_tensors('train', encoder_name=ENCODER_NAME)
val_loader = data.load_multimodal_tensors('val', encoder_name=ENCODER_NAME, shuffle=False)

val_participant_ids = data.load_multimodal_participant_ids('val')
VAL_PARTICIPANT_IDS = sorted(set(val_participant_ids))

assert len(val_participant_ids) == len(val_loader.dataset)

print('number of validation participants:', len(VAL_PARTICIPANT_IDS))
print('validation participant IDs:', VAL_PARTICIPANT_IDS)
print()
sample_batch = next(iter(train_loader))
print('batch shapes:', [tuple(x.shape) for x in sample_batch])
print('Diagnostic 3 uses train + validation only; the held-out test set is not loaded.')

number of validation participants: 6
validation participant IDs: [np.str_('CGMacros-005'), np.str_('CGMacros-016'), np.str_('CGMacros-027'), np.str_('CGMacros-044'), np.str_('CGMacros-046'), np.str_('CGMacros-048')]

batch shapes: [(256, 24), (256,), (256,), (256, 768), (256, 5)]
Diagnostic 3 uses train + validation only; the held-out test set is not loaded.


In [5]:
def forward_fn(model, batch, device):
    cgm, tsm, has_meal, img_feat, target = batch
    return (model(cgm.to(device), tsm.to(device), img_feat.to(device), has_meal.to(device)),
            target.to(device))

In [6]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
all_results = []

for seed in SEEDS:
    print(f"Validation results — {ENCODER_NAME} | seed={seed}\n")

    base_model = load_frozen_baseline(seed, EXP2_CHECKPOINT_DIR)

    train.set_seed(seed)
    model = PartiallyFrozenSharedHeadPredictor(base_model=base_model, img_feat_dim=IMG_FEAT_DIM,
                                                img_proj_dim=IMG_PROJ_DIM)

    model, history = train.train_model(
        model, train_loader, val_loader, stats['cgm_mean'], stats['cgm_std'],
        lr=0.0001, weight_decay=1e-05, forward_fn=forward_fn,
        checkpoint_dir=EXPERIMENT_DIR / 'checkpoints' / f'diagnostic3_partial_freeze_seed{seed}',
        verbose=False, progress_bar=True, progress_desc=f'seed={seed}',
    )
    n_epochs = len(history['train_loss'])
    val_loss_hist = history['val_loss']
    best_epoch = int(np.argmin(val_loss_hist) + 1)

    y_pred_val, y_true_val = train.predict(model, val_loader, forward_fn=forward_fn)
    val_results = metrics.evaluate(y_pred_val, y_true_val, stats['cgm_mean'], stats['cgm_std'])
    val_results_per_participant = metrics.evaluate_per_participant(
            y_pred_val, y_true_val, val_participant_ids,
            stats['cgm_mean'], stats['cgm_std'])

    seed_result = {
        'seed': seed, 'n_epochs_trained': n_epochs, 'best_epoch': best_epoch,
        'train_loss_history': [float(x) for x in history['train_loss']],
        'val_loss_history': [float(x) for x in val_loss_hist],
        'val_results': val_results,
        'val_results_per_participant': val_results_per_participant,
    }
    all_results.append(seed_result)

    with open(RESULTS_DIR / f'diagnostic3_partial_freeze_seed{seed}_results.json', 'w') as f:
        json.dump(seed_result, f, indent=2)

    val_rmse = [round(val_results[h]['rmse'], 2) for h in HORIZONS]
    print(f"seed={seed:>3}  stopped at epoch {n_epochs:>3}/50  best epoch={best_epoch:>3}  "
          f"validation RMSE @ {HORIZONS} min = {val_rmse}")

Validation results — clip | seed=7

device: cuda (NVIDIA RTX PRO 6000 Blackwell Server Edition)  model: PartiallyFrozenSharedHeadPredictor  params: 1,030,445


seed=7:   0%|          | 0/50 [00:00<?, ?it/s]

seed=  7  stopped at epoch  15/50  best epoch=  5  validation RMSE @ [15, 30, 60, 90, 120] min = [10.55, 16.61, 24.0, 28.48, 31.32]
Validation results — clip | seed=14

device: cuda (NVIDIA RTX PRO 6000 Blackwell Server Edition)  model: PartiallyFrozenSharedHeadPredictor  params: 1,030,445


seed=14:   0%|          | 0/50 [00:00<?, ?it/s]

seed= 14  stopped at epoch  16/50  best epoch=  6  validation RMSE @ [15, 30, 60, 90, 120] min = [10.54, 16.55, 23.85, 28.33, 31.17]
Validation results — clip | seed=21

device: cuda (NVIDIA RTX PRO 6000 Blackwell Server Edition)  model: PartiallyFrozenSharedHeadPredictor  params: 1,030,445


...

seed= 56  stopped at epoch  14/50  best epoch=  4  validation RMSE @ [15, 30, 60, 90, 120] min = [10.63, 16.56, 23.98, 28.49, 31.44]
Validation results — clip | seed=63

device: cuda (NVIDIA RTX PRO 6000 Blackwell Server Edition)  model: PartiallyFrozenSharedHeadPredictor  params: 1,030,445


seed=63:   0%|          | 0/50 [00:00<?, ?it/s]

seed= 63  stopped at epoch  14/50  best epoch=  4  validation RMSE @ [15, 30, 60, 90, 120] min = [10.57, 16.54, 23.87, 28.4, 31.34]
Validation results — clip | seed=70

device: cuda (NVIDIA RTX PRO 6000 Blackwell Server Edition)  model: PartiallyFrozenSharedHeadPredictor  params: 1,030,445


seed=70:   0%|          | 0/50 [00:00<?, ?it/s]

seed= 70  stopped at epoch  14/50  best epoch=  4  validation RMSE @ [15, 30, 60, 90, 120] min = [10.65, 16.54, 23.9, 28.46, 31.42]


In [7]:
best_epochs = [r['best_epoch'] for r in all_results]

per_horizon_rmse = {}

for h in HORIZONS:
    seed_values = [
        r['val_results'][h]['rmse']
        for r in all_results
    ]

    mean_rmse = float(np.mean(seed_values))
    sd1 = float(np.std(seed_values, ddof=1))

    participant_means = []

    for pid in VAL_PARTICIPANT_IDS:

        values_across_seeds = [
            r['val_results_per_participant'][pid][h]['rmse']
            for r in all_results
        ]

        participant_means.append(
            float(np.mean(values_across_seeds))
        )

    sd2 = float(
        np.std(participant_means, ddof=1)
    )

    per_horizon_rmse[h] = {
        'mean': mean_rmse,
        'sd1_across_seeds': sd1,
        'sd2_across_participants': sd2,
    }


print(
    '=== Diagnostic 3a: Partial Freeze, Shared Head '
    '(CLIP, img_proj_dim=4, train+val only) ==='
)

print(
    'RMSE is reported as mean +/- sd1 (sd2), '
    'where sd1 is across seeds and sd2 is across validation participants.'
)

print()

print(
    f"Best epoch = "
    f"{np.mean(best_epochs):.2f} +/- "
    f"{np.std(best_epochs, ddof=1):.2f}"
)

for h in HORIZONS:

    s = per_horizon_rmse[h]

    print(
        f"{h:>3}-min: "
        f"RMSE = {s['mean']:.2f} +/- "
        f"{s['sd1_across_seeds']:.2f} "
        f"({s['sd2_across_participants']:.2f}) mg/dL"
    )

=== Diagnostic 3a: Partial Freeze, Shared Head (CLIP, img_proj_dim=4, train+val only) ===
RMSE is reported as mean +/- sd1 (sd2), where sd1 is across seeds and sd2 is across validation participants.

Best epoch = 4.50 +/- 0.71
 15-min: RMSE = 10.57 +/- 0.04 (2.14) mg/dL
 30-min: RMSE = 16.52 +/- 0.06 (3.54) mg/dL
 60-min: RMSE = 23.85 +/- 0.09 (6.99) mg/dL
 90-min: RMSE = 28.38 +/- 0.08 (10.17) mg/dL
120-min: RMSE = 31.27 +/- 0.11 (12.04) mg/dL


In [8]:
with open(SUMMARY_PATH, 'w') as f:

    json.dump(
        {
            'seeds': SEEDS,
            'encoder': ENCODER_NAME,
            'img_proj_dim': IMG_PROJ_DIM,

            'best_epoch': {
                'mean': float(np.mean(best_epochs)),
                'sd': float(np.std(best_epochs, ddof=1))
            },

            'val_rmse': per_horizon_rmse,
        },
        f,
        indent=2
    )


print(f'\nsaved per-seed results to {RESULTS_DIR}')
print(f'saved summary to {SUMMARY_PATH}')


saved per-seed results to /content/drive/MyDrive/dissertation/experiments/4_diagnostic/3_diagnostic/results/diagnostic_3_clip
saved summary to /content/drive/MyDrive/dissertation/experiments/4_diagnostic/3_diagnostic/results/diagnostic_3_clip/diagnostic3_partial_freeze_shared_head_clip_summary.json


In [11]:
BASE_DIR = (
    PROJECT_ROOT / 'experiments' / '3_initial_multimodal' / 'multimodal'
    / 'results' / 'gated_cgm_time'
)
DIAG2_DIR = EXPERIMENT_DIR = PROJECT_ROOT / 'experiments' / '4_diagnostic' / '2_diagnostic' / 'results'


def _get_horizon_entry(results_dict, horizon):
    """Handle JSON dictionaries whose horizon keys may be strings or integers."""
    if str(horizon) in results_dict:
        return results_dict[str(horizon)]
    if horizon in results_dict:
        return results_dict[horizon]
    raise KeyError(f'Horizon {horizon} not found. Available keys: {list(results_dict.keys())}')


def load_seed_results(result_dir, stub, seeds, result_key='val_results'):
    """Load one result JSON per seed and return {seed: full_result_dict}."""
    loaded = {}

    for seed in seeds:
        path = result_dir / f'{stub}_seed{seed}_results.json'

        if not path.exists():
            raise FileNotFoundError(
                f'Missing result file for seed {seed}: {path}\n'
                f'Check result_dir/stub before running this comparison.'
            )

        with open(path) as f:
            result = json.load(f)

        if result_key not in result:
            raise KeyError(
                f'{path.name} does not contain "{result_key}". '
                f'Available keys: {list(result.keys())}'
            )

        loaded[seed] = result

    return loaded


def rmse_by_seed(loaded_results, seeds, horizons, result_key='val_results'):
    """Return {horizon: np.array([RMSE for each seed in SEEDS order])}."""
    out = {}

    for h in horizons:
        vals = []
        for seed in seeds:
            entry = _get_horizon_entry(loaded_results[seed][result_key], h)
            vals.append(float(entry['rmse']))
        out[h] = np.asarray(vals, dtype=float)

    return out


base_loaded = load_seed_results(
    BASE_DIR,
    'gated_cgm_time',
    SEEDS,
    result_key='val_results',
)

joint_loaded = load_seed_results(
    DIAG2_DIR,
    'diagnostic2_proj4',
    SEEDS,
    result_key='val_results',
)

base_by_h = rmse_by_seed(base_loaded, SEEDS, HORIZONS)
joint_by_h = rmse_by_seed(joint_loaded, SEEDS, HORIZONS)

partial_by_h = {
    h: np.asarray(
        [float(_get_horizon_entry(r['val_results'], h)['rmse']) for r in all_results],
        dtype=float,
    )
    for h in HORIZONS
}

current_seeds = [int(r['seed']) for r in all_results]
if current_seeds != SEEDS:
    raise ValueError(
        f'Current Diagnostic 3 seed order does not match SEEDS.\n'
        f'Expected: {SEEDS}\n'
        f'Found:    {current_seeds}'
    )


print('=== Paired validation RMSE across seeds ===')
print(
    f"{'horizon':>9}"
    f"{'CGM+time':>19}"
    f"{'fully joint d=4':>22}"
    f"{'partial freeze d=4':>24}"
)
for h in HORIZONS:
    base = base_by_h[h]
    joint = joint_by_h[h]
    partial = partial_by_h[h]

    print(
        f"{h:>6} min"
        f"{np.mean(base):>10.3f} +/- {np.std(base, ddof=1):<6.3f}"
        f"{np.mean(joint):>13.3f} +/- {np.std(joint, ddof=1):<6.3f}"
        f"{np.mean(partial):>15.3f} +/- {np.std(partial, ddof=1):<6.3f}"
    )


print('\n=== Diagnostic 3 paired effect: Partial Freeze vs CGM+Time ===')
print(
    f"{'horizon':>9}"
    f"{'delta RMSE':>22}"
    f"{'improvement %':>24}"
    f"{'improved seeds':>18}"
)
for h in HORIZONS:
    base = base_by_h[h]
    partial = partial_by_h[h]
    delta = partial - base
    improvement_pct = (base - partial) / base * 100.0

    wins = int(np.sum(partial < base))
    ties = int(np.sum(np.isclose(partial, base, atol=1e-12)))
    losses = len(SEEDS) - wins - ties

    print(
        f"{h:>6} min"
        f"{np.mean(delta):>11.4f} +/- {np.std(delta, ddof=1):<7.4f}"
        f"{np.mean(improvement_pct):>13.3f} +/- {np.std(improvement_pct, ddof=1):<7.3f}"
        f"{wins:>8}/{len(SEEDS)}"
        f"  (ties={ties}, losses={losses})"
    )


print('\n=== Effect of freezing: Partial Freeze vs Fully Joint (both d=4) ===')
print(
    f"{'horizon':>9}"
    f"{'delta RMSE':>22}"
    f"{'improvement %':>24}"
    f"{'PF better seeds':>18}"
)
for h in HORIZONS:
    joint = joint_by_h[h]
    partial = partial_by_h[h]

    delta = partial - joint
    improvement_pct = (joint - partial) / joint * 100.0

    wins = int(np.sum(partial < joint))
    ties = int(np.sum(np.isclose(partial, joint, atol=1e-12)))
    losses = len(SEEDS) - wins - ties

    print(
        f"{h:>6} min"
        f"{np.mean(delta):>11.4f} +/- {np.std(delta, ddof=1):<7.4f}"
        f"{np.mean(improvement_pct):>13.3f} +/- {np.std(improvement_pct, ddof=1):<7.3f}"
        f"{wins:>8}/{len(SEEDS)}"
        f"  (ties={ties}, losses={losses})"
    )

rows = []

for h in HORIZONS:
    for i, seed in enumerate(SEEDS):
        base = base_by_h[h][i]
        joint = joint_by_h[h][i]
        partial = partial_by_h[h][i]

        rows.append({
            'horizon_min': h,
            'seed': seed,
            'cgm_time_rmse': base,
            'fully_joint_dim4_rmse': joint,
            'partial_freeze_dim4_rmse': partial,
            'partial_minus_base': partial - base,
            'partial_minus_joint': partial - joint,
            'partial_improvement_vs_base_pct': (base - partial) / base * 100.0,
        })


=== Paired validation RMSE across seeds ===
  horizon           CGM+time       fully joint d=4      partial freeze d=4
    15 min    10.576 +/- 0.070        10.705 +/- 0.112          10.573 +/- 0.040 
    30 min    16.680 +/- 0.070        16.848 +/- 0.095          16.522 +/- 0.062 
    60 min    24.069 +/- 0.056        24.331 +/- 0.111          23.848 +/- 0.091 
    90 min    28.401 +/- 0.048        28.789 +/- 0.173          28.377 +/- 0.080 
   120 min    31.246 +/- 0.059        31.532 +/- 0.155          31.265 +/- 0.108 

=== Diagnostic 3 paired effect: Partial Freeze vs CGM+Time ===
  horizon            delta RMSE           improvement %    improved seeds
    15 min    -0.0035 +/- 0.0756         0.029 +/- 0.714         6/10  (ties=0, losses=4)
    30 min    -0.1586 +/- 0.0723         0.950 +/- 0.431        10/10  (ties=0, losses=0)
    60 min    -0.2215 +/- 0.0848         0.920 +/- 0.352        10/10  (ties=0, losses=0)
    90 min    -0.0247 +/- 0.0645         0.087 +/- 0.227       